# Chess Neural Network — Training Pipeline

Auto-detects training phase based on available checkpoints.
Runs on Kaggle T4/P100 GPU.

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/amiohossain/Chess.git /kaggle/working/Chess
!pip install -q -r /kaggle/working/Chess/requirements.txt
!pip install -q zstandard  # For zstd decompression fallback

In [ ]:
import os
import sys
import glob
import subprocess
import json

sys.path.insert(0, '/kaggle/working/Chess')

DATA_DIR = '/kaggle/input/chess-training-data'
HDF5_PATH = '/kaggle/working/supervised_positions.h5'
os.makedirs('/kaggle/working/data', exist_ok=True)

# Debug: list input contents
print('=== Dataset contents ===')
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        path = os.path.join(root, f)
        print(f'  {path}  ({os.path.getsize(path)/1e6:.1f} MB)')
print('=======================')

# Check if HDF5 already exists in the input dataset
h5_files = glob.glob(f'{DATA_DIR}/**/*.h5', recursive=True)
if h5_files:
    print(f'Found existing HDF5: {h5_files[0]}')
    !cp {h5_files[0]} {HDF5_PATH}
else:
    print('No HDF5 found. Looking for PGN.zst files...')
    zst_files = sorted(glob.glob(f'{DATA_DIR}/**/*.pgn.zst', recursive=True))
    zst_files += sorted(glob.glob(f'{DATA_DIR}/**/*.zst', recursive=True))
    zst_files = sorted(set(zst_files))
    print(f'Found {len(zst_files)} zst files')

    if not zst_files:
        print('ERROR: No zst files found!')
        print('Files at root of dataset:')
        !ls -la {DATA_DIR}/
    else:
        # Decompress all PGNs, concatenate, then process once
        combined_pgn = '/kaggle/working/combined.pgn'

        for i, zst in enumerate(zst_files):
            pgn_part = f'/kaggle/working/temp_{i}.pgn'
            print(f'[{i+1}/{len(zst_files)}] Decompressing {os.path.basename(zst)}...')
            result = subprocess.run(['zstd', '-d', '-f', zst, '-o', pgn_part],
                                   capture_output=True, text=True)
            if result.returncode != 0:
                print(f'  zstd failed: {result.stderr}')
                # Try with pyzstd as fallback
                import zstandard as zstd_lib
                with open(zst, 'rb') as src, open(pgn_part, 'wb') as dst:
                    dctx = zstd_lib.ZstdDecompressor()
                    dctx.copy_stream(src, dst)

            # Concatenate to combined file
            with open(pgn_part, 'r', errors='replace') as src:
                with open(combined_pgn, 'a' if i > 0 else 'w') as dst:
                    dst.write(src.read())
            os.remove(pgn_part)
            print(f'  Done ({os.path.getsize(combined_pgn)/1e6:.1f} MB so far)')

        if os.path.exists(combined_pgn):
            print(f'Combined PGN size: {os.path.getsize(combined_pgn) / 1e6:.0f} MB')

            # Process combined PGN to HDF5
            from src.data.pgn_processor import process_pgn
            n = process_pgn(
                pgn_path=combined_pgn,
                output_path=HDF5_PATH,
                max_positions=5_000_000,
                min_elo=2200,
            )
            os.remove(combined_pgn)
            print(f'HDF5 created with {n} positions')
        else:
            print('ERROR: combined.pgn was not created')

In [ ]:
# Run the training pipeline (auto-detects phase)
from src.kaggle.kaggle_main import run_session
run_session()